# INSTALACION DE LOS DOCUMENTOS

## Hugging Face

In [ ]:
!pip install -q -U huggingface_hub pysam pandas matplotlib seaborn python-docx

In [ ]:
# Antes de compilar esta celda, entrar a su cuenta de Hugging Face (tienen que tener los permisos necesarios)
from huggingface_hub import login

login()
#Despues de compilar aparecera un link, entraran y colocaran los 8 digitos del codigo que aparece
#luego daran permitir acceso, y se conectara a su cuenta ya existente y con los permisos necesarios

In [ ]:
from huggingface_hub import whoami

info = whoami()

print(info)

## Lectura de los archivos que nos brindaron

In [ ]:
REPO_ID = "SageBio/mva-hackathon-2026-data"
REPO_TYPE = "dataset"

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

archivos = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE
)

print(f"Cantidad de archivos: {len(archivos)}")

for archivo in archivos:
    print(archivo)

In [ ]:
import pandas as pd

registros = []

for archivo in archivos:

    nombre = archivo.lower()

    if nombre.endswith(".vcf.gz.tbi"):
        tipo = "Índice VCF"

    elif nombre.endswith(".vcf.gz"):
        tipo = "VCF"

    elif name := nombre.endswith(".fastq.gz"):
        tipo = "FASTQ"

    elif nombre.endswith(".docx"):
        tipo = "Documento Word"

    elif nombre.endswith(".md"):
        tipo = "README"

    else:
        tipo = "Otro"

    registros.append({
        "archivo": archivo,
        "tipo": tipo
    })

inventario = pd.DataFrame(registros)

display(inventario)

## Lectura de los archivos del concurso

In [ ]:
# Descarga del archivo README
from huggingface_hub import hf_hub_download

ruta_readme = hf_hub_download(
    repo_id=REPO_ID,
    filename="README.md",
    repo_type=REPO_TYPE
)

print("Archivo descargado:")
print(ruta_readme)

In [ ]:
# Lectura del README
with open(ruta_readme, "r", encoding="utf-8") as f:
    readme = f.read()

print(readme)

In [ ]:
# Descarga del documento del RETO
ruta_docx = hf_hub_download(
    repo_id=REPO_ID,
    filename="Challenge_Clinical_Phenotype_1.docx",
    repo_type=REPO_TYPE
)

print(ruta_docx)

In [ ]:
# Lectura del Reto
from docx import Document

documento = Document(ruta_docx)

texto = []

for parrafo in documento.paragraphs:
    texto.append(parrafo.text)

texto_completo = "\n".join(texto)

print(texto_completo)

## Archivos de VCF

In [ ]:
archivo_vcf = "WGS_EX2312012_HGWCNDSX7.vcf.gz"

ruta_vcf = hf_hub_download(
    repo_id=REPO_ID,
    filename=archivo_vcf,
    repo_type=REPO_TYPE
)

print("VCF:")
print(ruta_vcf)

In [ ]:
archivo_tbi = "WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi"

ruta_tbi = hf_hub_download(
    repo_id=REPO_ID,
    filename=archivo_tbi,
    repo_type=REPO_TYPE
)

print("Índice:")
print(ruta_tbi)

In [ ]:
# Celda para abrir los archivos VCF
import pysam

vcf = pysam.VariantFile(ruta_vcf)

print(vcf)

### Mostrando el VCF

In [ ]:
print("Número de muestras:", len(vcf.header.samples))

print("\nMuestras:")
for sample in vcf.header.samples:
    print(sample)

In [ ]:
#
print("Cromosomas disponibles:")

for cromosoma in vcf.header.contigs:
    print(cromosoma)

In [ ]:
print("FORMATO DEL VCF:")
print(vcf.header.version)

print("\nMUESTRAS:")
print(list(vcf.header.samples))

print("\nCONTIGS:")
print(list(vcf.header.contigs)[:20])

In [ ]:
print("VARIABLES INFO")
print("=" * 60)

for key, item in vcf.header.info.items():
    print(f"{key:25} | Tipo: {item.type:8} | Número: {item.number}")
    print(f"{'':25} | {item.description}")
    print()

In [ ]:
print("VARIABLES FORMAT")
print("=" * 60)

for key, item in vcf.header.formats.items():
    print(f"{key:15} | Tipo: {item.type:8} | Número: {item.number}")
    print(f"{'':15} | {item.description}")
    print()

In [ ]:
for i, record in enumerate(vcf):
    print(record)

    if i == 9:
        break

In [ ]:
import pandas as pd

vcf.reset()

registros = []

for i, record in enumerate(vcf):

    registros.append({
        "CHROM": record.chrom,
        "POS": record.pos,
        "ID": record.id,
        "REF": record.ref,
        "ALT": ",".join(record.alts) if record.alts else None,
        "QUAL": record.qual,
        "FILTER": ",".join(record.filter.keys())
    })

    if i >= 999:
        break

df_variantes = pd.DataFrame(registros)

df_variantes.head()

In [ ]:
df_variantes["CHROM"].value_counts()

In [ ]:
df_variantes["REF"].value_counts()

In [ ]:
df_variantes["ALT"].value_counts()

In [ ]:
df_variantes["FILTER"].value_counts()

In [ ]:
df_variantes["CAMBIO"] = (
    df_variantes["REF"] + " → " + df_variantes["ALT"]
)

df_variantes["CAMBIO"].value_counts().head(20)

In [ ]:
def clasificar_variante(row):
    ref = row["REF"]
    alt = row["ALT"].split(",")[0]

    if len(ref) == 1 and len(alt) == 1:
        return "SNV"
    elif len(ref) < len(alt):
        return "Inserción"
    elif len(ref) > len(alt):
        return "Deleción"
    else:
        return "Otra"

df_variantes["TIPO_VARIANTE"] = df_variantes.apply(
    clasificar_variante,
    axis=1
)

df_variantes["TIPO_VARIANTE"].value_counts()

In [ ]:
for columna in df_variantes.columns:
    print("\n" + "="*60)
    print("VARIABLE:", columna)
    print("="*60)
    print("Número de valores diferentes:",
          df_variantes[columna].nunique())

    print("\nCategorías / valores más frecuentes:")
    print(df_variantes[columna].value_counts().head(15))

In [ ]:
resumen = pd.DataFrame({
    "Variable": [
        "CHROM",
        "POS",
        "ID",
        "REF",
        "ALT",
        "QUAL",
        "FILTER"
    ],
    "Tipo": [
        "Categórica",
        "Numérica",
        "Categórica",
        "Categórica",
        "Categórica",
        "Numérica",
        "Categórica"
    ],
    "N_unicos": [
        df_variantes["CHROM"].nunique(),
        df_variantes["POS"].nunique(),
        df_variantes["ID"].nunique(),
        df_variantes["REF"].nunique(),
        df_variantes["ALT"].nunique(),
        df_variantes["QUAL"].nunique(),
        df_variantes["FILTER"].nunique()
    ]
})

resumen

In [ ]:
for i, variante in enumerate(vcf):

    print(
        f"{variante.chrom}:{variante.pos}",
        "REF =", variante.ref,
        "ALT =", variante.alts,
        "QUAL =", variante.qual
    )

    if i == 9:
        break

In [ ]:
print("INFO:")
for key, value in vcf.header.info.items():
    print(key, "->", value)

print("\nFORMAT:")
for key, value in vcf.header.formats.items():
    print(key, "->", value)